In [1]:
from pathlib import Path
import csv
import json
import shutil

LAST_NAME = "Nicolas"
STUDENT_ID = "TUPM-26-7931"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)

Student: Nicolas
Student ID: TUPM-26-7931
Seed Digit: 1
ID Sum: 28
Name Length: 7


In [2]:
WORK_DIR = Path(".")
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

dataset_path = WORK_DIR / "user_behavior_dataset.csv"

print("Workspace:", WORK_DIR.resolve())
print("Dataset exists:", dataset_path.exists())
print("Output folder exists:", OUTPUT_DIR.exists())
print("Backup folder exists:", BACKUP_DIR.exists())

Workspace: C:\Users\Chichi\Lab6_File_Handling_Nicolas\Lab6_File_Handling_Additional_Exercises\Exercise_2_Mobile_Device_Usage
Dataset exists: True
Output folder exists: True
Backup folder exists: True


In [3]:
TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"

TARGET_BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1

print("Target Operating System:", TARGET_OS)
print("Target Behavior Class:", TARGET_BEHAVIOR_CLASS)

Target Operating System: iOS
Target Behavior Class: 2


In [4]:
with open(dataset_path, "r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)

    print("Dataset Columns:")
    print(reader.fieldnames)

    print("\nSample Records:")

    for i, record in enumerate(reader):
        if i == 3:
            break
        print(record)

Dataset Columns:
['User ID', 'Device Model', 'Operating System', 'App Usage Time (min/day)', 'Screen On Time (hours/day)', 'Battery Drain (mAh/day)', 'Number of Apps Installed', 'Data Usage (MB/day)', 'Age', 'Gender', 'User Behavior Class']

Sample Records:
{'User ID': '1', 'Device Model': 'Google Pixel 5', 'Operating System': 'Android', 'App Usage Time (min/day)': '393', 'Screen On Time (hours/day)': '6.4', 'Battery Drain (mAh/day)': '1872', 'Number of Apps Installed': '67', 'Data Usage (MB/day)': '1122', 'Age': '40', 'Gender': 'Male', 'User Behavior Class': '4'}
{'User ID': '2', 'Device Model': 'OnePlus 9', 'Operating System': 'Android', 'App Usage Time (min/day)': '268', 'Screen On Time (hours/day)': '4.7', 'Battery Drain (mAh/day)': '1331', 'Number of Apps Installed': '42', 'Data Usage (MB/day)': '944', 'Age': '47', 'Gender': 'Female', 'User Behavior Class': '3'}
{'User ID': '3', 'Device Model': 'Xiaomi Mi 11', 'Operating System': 'Android', 'App Usage Time (min/day)': '154', 'Scre

In [6]:
selected_users = []

total_records = 0
matching_os_records = 0
invalid_records = 0

total_battery_drain = 0.0
total_data_usage = 0.0

valid_battery_records = 0
valid_data_records = 0

required_columns = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)",
    "User Behavior Class"
]

with open(dataset_path, "r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)

    if not all(col in reader.fieldnames for col in required_columns):
        raise ValueError(
            f"Missing required columns. Available: {reader.fieldnames}"
        )

    for record in reader:
        total_records += 1

        try:
            operating_system = record["Operating System"].strip()

            if operating_system.lower() != TARGET_OS.lower():
                continue

            matching_os_records += 1

            behavior_class = int(record["User Behavior Class"])

            if behavior_class != TARGET_BEHAVIOR_CLASS:
                continue

            battery_drain = float(record["Battery Drain (mAh/day)"])
            data_usage = float(record["Data Usage (MB/day)"])

            if battery_drain < 0 or data_usage < 0:
                raise ValueError("Negative measurement")

            selected_users.append({
                "User ID": record["User ID"].strip(),
                "Device Model": record["Device Model"].strip(),
                "Operating System": operating_system,
                "Battery Drain (mAh/day)": battery_drain,
                "Data Usage (MB/day)": data_usage
            })

            total_battery_drain += battery_drain
            total_data_usage += data_usage

            valid_battery_records += 1
            valid_data_records += 1

        except (ValueError, TypeError, KeyError):
            invalid_records += 1

print("Total records:", total_records)
print("Records matching OS:", matching_os_records)
print("Records in target behavior class:", len(selected_users))
print("Invalid records:", invalid_records)

Total records: 700
Records matching OS: 146
Records in target behavior class: 32
Invalid records: 0


In [7]:
average_battery_drain = (
    total_battery_drain / valid_battery_records
    if valid_battery_records else 0
)

average_data_usage = (
    total_data_usage / valid_data_records
    if valid_data_records else 0
)

print("Average Battery Drain:",
      round(average_battery_drain, 2), "mAh/day")

print("Average Data Usage:",
      round(average_data_usage, 2), "MB/day")

Average Battery Drain: 872.03 mAh/day
Average Data Usage: 451.34 MB/day


In [8]:
csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_profile.csv"

fieldnames = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)"
]

with open(csv_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(selected_users)

print("Created CSV:", csv_path)
print("Exported users:", len(selected_users))

Created CSV: output\Nicolas_TUPM-26-7931_mobile_profile.csv
Exported users: 32


In [9]:
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_summary.json"

summary_data = {
    "last_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "id_sum": ID_SUM,
    "target_operating_system": TARGET_OS,
    "target_behavior_class": TARGET_BEHAVIOR_CLASS,
    "total_records": total_records,
    "matching_os_records": matching_os_records,
    "target_class_records": len(selected_users),
    "invalid_records": invalid_records,
    "average_battery_drain": round(average_battery_drain, 2),
    "average_data_usage": round(average_data_usage, 2),
    "exported_csv": csv_path.name
}

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary_data, file, indent=4)

print("JSON summary created:", json_path)

JSON summary created: output\Nicolas_TUPM-26-7931_mobile_summary.json


In [10]:
backup_path = BACKUP_DIR / csv_path.name

shutil.copy2(csv_path, backup_path)

print("Original CSV:", csv_path)
print("Backup CSV:", backup_path)
print("Backup exists:", backup_path.exists())
print("Backup size:", backup_path.stat().st_size, "bytes")

Original CSV: output\Nicolas_TUPM-26-7931_mobile_profile.csv
Backup CSV: backup\Nicolas_TUPM-26-7931_mobile_profile.csv
Backup exists: True
Backup size: 1082 bytes


In [11]:
with open(csv_path, "r", encoding="utf-8", newline="") as file:
    exported_count = sum(1 for row in csv.DictReader(file))

with open(json_path, "r", encoding="utf-8") as file:
    loaded_summary = json.load(file)

print("EXERCISE 2 VERIFICATION")

print("Selected users:", len(selected_users))
print("Exported CSV records:", exported_count)

print("JSON loaded:",
      loaded_summary["student_id"] == STUDENT_ID)

print("Count verification:",
      exported_count == len(selected_users))

print("Backup exists:", backup_path.exists())

EXERCISE 2 VERIFICATION
Selected users: 32
Exported CSV records: 32
JSON loaded: True
Count verification: True
Backup exists: True
